# 3D roof reconstruction benchmark

This research notebook compares bounded diagnostic plane evidence with complete reconstruction backends. RANSAC is not a complete LoD2 backend. Normal production requests route to one backend; running several methods is restricted to benchmark or research mode.

**Relationship to notebook 01:** notebook 01 validates the input evidence; this notebook evaluates reconstruction behavior. Core RANSAC is invoked first as production pre-routing evidence. The later surface-building experiment remains an explicitly independent research baseline.

**Current checkpoint:** acquisition, LiDAR QC, roof complexity, LoD feasibility, routing, Roofer and City3D execution, shared observation filtering, structural QC, and point-to-surface QC are implemented. The next benchmark block is multi-building routing calibration.


## Reproduce the benchmark inputs

Change `CADASTRAL_ROOT_ID` and run from the beginning. This notebook is independent of notebook 01 and prepares its own Catastro, LiDAR, and Roofer artifacts.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import laspy
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from matplotlib.colors import to_hex
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree
from shapely import MultiPoint, Polygon, concave_hull
from shapely.ops import triangulate, unary_union

from urbanstock3d.processors.lidar import points_in_polygon
from urbanstock3d.providers.pnoa_lidar import footprint_polygons_utm
from urbanstock3d.providers.roofer import RooferClient
from urbanstock3d.reconstruction.execution import execute_reconstruction_plan
from urbanstock3d.reconstruction.execution import finalize_or_execute_fallback
from urbanstock3d.reconstruction.models import (
    GeometryProvenance,
    ReconstructionEvidence,
    ReconstructionRequest,
    ReconstructionResult,
)
from urbanstock3d.reconstruction.enums import BackendName, ReconstructionStatus
from urbanstock3d.reconstruction.backends import BackendRegistry, RooferBackend
from urbanstock3d.reconstruction.benchmark import (
    build_benchmark_entry,
    compare_reconstructions,
)
from urbanstock3d.reconstruction.planner import plan_reconstruction
from urbanstock3d.reconstruction.quality.lidar import assess_lidar_quality
from urbanstock3d.reconstruction.quality.lod_feasibility import (
    LodEvidenceAvailability,
    assess_lod_feasibility,
)
from urbanstock3d.reconstruction.quality.ransac import RansacParameters, detect_roof_planes
from urbanstock3d.reconstruction.quality.roof_complexity import assess_roof_complexity
from urbanstock3d.reconstruction.validation import (
    assess_cityjson_lidar_fit,
    assess_obj_lidar_fit,
    evaluate_reconstruction_quality,
    read_obj,
    select_roof_observations,
    validate_cityjsonseq,
    validate_obj,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUN_BENCHMARK_PIPELINE = True
RUN_PLANNED_RECONSTRUCTION = False
RUN_QUALITY_FALLBACK = False
# Change only this 14-character cadastral root to benchmark another building.
CADASTRAL_ROOT_ID = "4531917YJ2743B"
assert len(CADASTRAL_ROOT_ID) == 14, "Expected a 14-character cadastral root"
BUILDING_ID = f"ES.SDGC.BU.{CADASTRAL_ROOT_ID}"
BUILDING_PATH = PROJECT_ROOT / "outputs" / BUILDING_ID / "building.geojson"
CROP_PATH = PROJECT_ROOT / "outputs" / BUILDING_ID / "lidar_context_crop.laz"
ROOFER_ROOT = PROJECT_ROOT / "outputs" / BUILDING_ID / "roofer"
ROOFER_EXECUTABLE = PROJECT_ROOT / ".tools" / "roofer-v1.0.0" / "bin" / "roofer.exe"

In [ ]:
# Benchmark mode deliberately prepares Roofer and the shared evidence from a clean state.
if RUN_BENCHMARK_PIPELINE:
    scripts_directory = PROJECT_ROOT / "scripts"
    urbanstock_executable = Path(sys.executable).with_name(
        "urbanstock.exe" if os.name == "nt" else "urbanstock"
    )
    commands = [
        [
            str(urbanstock_executable),
            "resolve",
            "--refcat",
            CADASTRAL_ROOT_ID,
            "--output-dir",
            str(PROJECT_ROOT / "outputs"),
        ],
        [
            sys.executable,
            str(scripts_directory / "process_lidar_crop.py"),
            str(BUILDING_PATH),
            "--buffer-m",
            "25",
            "--save-crop",
            str(CROP_PATH),
            "--output",
            str(PROJECT_ROOT / "outputs" / BUILDING_ID / "lidar_crop_audit.json"),
        ],
        [
            sys.executable,
            str(scripts_directory / "prepare_roofer_input.py"),
            str(BUILDING_PATH),
            str(ROOFER_ROOT / "footprint_25830.geojson"),
        ],
        [
            sys.executable,
            str(scripts_directory / "run_roofer.py"),
            str(CROP_PATH),
            str(ROOFER_ROOT / "footprint_25830.geojson"),
            str(ROOFER_ROOT / "native_cityjson"),
            "--executable",
            str(ROOFER_EXECUTABLE),
            "--jobs",
            "1",
        ],
    ]
    for command in commands:
        print(f"Running: {' '.join(command)}")
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print("Benchmark preparation skipped; existing artifacts will be used.")

## Load the common reconstruction evidence


In [ ]:
assert CROP_PATH.exists(), f"Missing LiDAR crop: {CROP_PATH}"
assert BUILDING_PATH.exists(), f"Missing building geometry: {BUILDING_PATH}"

cloud = laspy.read(CROP_PATH)
x = np.asarray(cloud.x)
y = np.asarray(cloud.y)
z = np.asarray(cloud.z)
classification = np.asarray(cloud.classification, dtype=np.uint8)
ground = classification == 2
assert ground.any(), "The context contains no classified ground points"
ground_z = float(np.median(z[ground]))
height = z - ground_z

building = json.loads(BUILDING_PATH.read_text(encoding="utf-8"))
footprint_polygons = footprint_polygons_utm(building["geometry"])
in_footprint = np.zeros(x.shape, dtype=bool)
for rings in footprint_polygons:
    in_footprint |= points_in_polygon(x, y, rings)

roof = in_footprint & (classification == 6)
roof_x = x[roof]
roof_y = y[roof]
roof_height = height[roof]
roof_points = np.column_stack((roof_x, roof_y, roof_height))
assert len(roof_points), "The footprint contains no classified roof points"

print(f"Benchmark building: {BUILDING_ID}")
print(f"Roof points: {len(roof_points):,}")
print(f"Ground reference: {ground_z:.2f} m")

## Select comparable roof observations

Class 6 means *building*, not necessarily visible roof. A shared local upper-envelope filter removes low facade or occluded returns before measuring either backend. Sparse neighborhoods are retained, and the audit counts below make the transformation explicit. This is evaluation preprocessing; it does not alter the reconstruction input.


In [ ]:
observation_mask, observation_report = select_roof_observations(roof_points)
print(json.dumps(observation_report.to_dict(), indent=2))

observation_figure = go.Figure()
for selected, label, color, opacity in (
    (False, "Rejected low building returns", "#d97706", 0.55),
    (True, "Selected roof observations", "#1f77b4", 0.85),
):
    candidate = roof_points[observation_mask == selected]
    observation_figure.add_trace(
        go.Scatter3d(
            x=candidate[:, 0], y=candidate[:, 1], z=candidate[:, 2],
            mode="markers", name=label, opacity=opacity,
            marker={"size": 2, "color": color},
        )
    )
observation_figure.update_layout(
    title="Shared LiDAR observations used by every backend metric", height=700,
    scene={"aspectmode": "data", "xaxis_title": "Easting (m)",
           "yaxis_title": "Northing (m)",
           "zaxis_title": "Height above local ground (m)"},
)
observation_figure.show()

## Production pre-routing evidence

Invoke the tested bounded RANSAC implementation before reconstruction. The report measures plane support, residuals, slopes and dominant normal groups; it does not generate roof polygons.


In [ ]:
core_ransac_parameters = RansacParameters()
core_plane_evidence = detect_roof_planes(roof_points, parameters=core_ransac_parameters)
print(json.dumps(core_plane_evidence.to_dict(), indent=2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
plane_support = [plane.support_ratio for plane in core_plane_evidence.planes]
plane_slopes = [plane.slope_deg for plane in core_plane_evidence.planes]
axes[0].bar(range(1, len(plane_support) + 1), plane_support, color="#1f77b4")
axes[0].set(title="Support per preliminary plane", xlabel="Plane", ylabel="Fraction of sampled roof points")
axes[1].bar(range(1, len(plane_slopes) + 1), plane_slopes, color="#d62728")
axes[1].set(title="Plane slopes", xlabel="Plane", ylabel="Slope (degrees)")
plt.show()


## Route one production reconstruction

The planner combines measured quality, complexity, LoD feasibility and the actually available native Roofer adapter. It selects one primary execution and at most one lower-LoD fallback; this cell does not run reconstruction.


In [ ]:
lidar_quality, _ = assess_lidar_quality(CROP_PATH, footprint_polygons)
building_parts_path = PROJECT_ROOT / "outputs" / BUILDING_ID / "building_parts.geojson"
building_part_count = len(json.loads(building_parts_path.read_text(encoding="utf-8"))["features"])
roof_complexity = assess_roof_complexity(
    footprint_polygons, core_plane_evidence, building_part_count=building_part_count
)
lod_feasibility = assess_lod_feasibility(
    LodEvidenceAvailability(True, bool(ground.any()), bool(roof.any())),
    lidar_quality,
    roof_complexity,
)
backend_registry = BackendRegistry()
backend_registry.register(
    RooferBackend(RooferClient(str(ROOFER_EXECUTABLE)), ROOFER_ROOT / "planned_runs")
)
reconstruction_plan = plan_reconstruction(
    ReconstructionRequest(),
    lidar_quality,
    roof_complexity,
    lod_feasibility,
    backend_registry,
)
routing_summary = {
    "requested_lod": reconstruction_plan.requested_lod.value,
    "target_lod": reconstruction_plan.target_lod.value if reconstruction_plan.target_lod else None,
    "backend": reconstruction_plan.selected_backend.value if reconstruction_plan.selected_backend else None,
    "profile": reconstruction_plan.selected_profile,
    "fallback_lod": reconstruction_plan.fallback_lod.value if reconstruction_plan.fallback_lod else None,
    "reasons": reconstruction_plan.reasons,
}
print(json.dumps(routing_summary, indent=2))


## Execute the selected production plan (optional)

Set `RUN_PLANNED_RECONSTRUCTION=True` only when you want to launch the selected backend. The executor runs exactly one primary backend. Fallback execution remains disabled until independent output QC exists.


In [ ]:
if RUN_PLANNED_RECONSTRUCTION:
    reconstruction_evidence = ReconstructionEvidence(
        building_id=BUILDING_ID,
        footprint=ROOFER_ROOT / "footprint_25830.geojson",
        lidar_points=CROP_PATH,
    )
    reconstruction_result = execute_reconstruction_plan(
        reconstruction_plan, reconstruction_evidence, backend_registry
    )
    execution_summary = {
        "status": reconstruction_result.status.value,
        "requested_lod": reconstruction_result.requested_lod.value,
        "targeted_lod": reconstruction_result.targeted_lod.value if reconstruction_result.targeted_lod else None,
        "delivered_lod": reconstruction_result.delivered_lod.value if reconstruction_result.delivered_lod else None,
        "backend": reconstruction_result.backend.value if reconstruction_result.backend else None,
        "model_path": str(reconstruction_result.model_path) if reconstruction_result.model_path else None,
        "reasons": reconstruction_result.reasons,
    }
    print(json.dumps(execution_summary, indent=2))
else:
    print("Planned execution skipped. Existing benchmark output is visualized below.")


## 1. Benchmark-only local feature experiment

A point normal and geometric descriptors are estimated from the covariance eigenvalues of each point's spherical neighborhood. Several radii are compared first because the radius controls the balance between noise sensitivity and loss of small roof structures. Only classified building points inside the cadastral footprint participate in this experiment.

In [ ]:
def compute_local_features(points, radius, min_neighbors=6):
    """Estimate covariance-based features for every sufficiently supported point."""
    tree = cKDTree(points)
    neighborhoods = tree.query_ball_point(points, radius)
    point_count = len(points)
    neighbor_count = np.fromiter(
        (len(neighbors) for neighbors in neighborhoods),
        dtype=np.int32,
        count=point_count,
    )
    normals = np.full((point_count, 3), np.nan)
    eigenvalues = np.full((point_count, 3), np.nan)

    for point_index, neighbor_indices in enumerate(neighborhoods):
        if len(neighbor_indices) < min_neighbors:
            continue
        neighbors = points[neighbor_indices]
        centered = neighbors - neighbors.mean(axis=0)
        covariance = centered.T @ centered / len(neighbors)
        values, vectors = np.linalg.eigh(covariance)
        values = np.maximum(values, 0.0)
        normal = vectors[:, 0]
        normals[point_index] = normal if normal[2] >= 0 else -normal
        eigenvalues[point_index] = values

    smallest, middle, largest = eigenvalues.T
    safe_largest = np.where(largest > 0, largest, np.nan)
    eigenvalue_sum = eigenvalues.sum(axis=1)
    return {
        "neighbor_count": neighbor_count,
        "valid": np.isfinite(largest),
        "normals": normals,
        "linearity": (largest - middle) / safe_largest,
        "planarity": (middle - smallest) / safe_largest,
        "scattering": smallest / safe_largest,
        "surface_variation": smallest / np.where(eigenvalue_sum > 0, eigenvalue_sum, np.nan),
    }


roof_points = np.column_stack((x[roof], y[roof], z[roof]))
candidate_radii = (0.75, 1.0, 1.5, 2.0, 2.5)
radius_comparison = {}

for radius in candidate_radii:
    candidate = compute_local_features(roof_points, radius)
    valid = candidate["valid"]
    radius_comparison[radius] = candidate
    print(
        f"Radius {radius:.2f} m | valid {valid.mean():.1%} | "
        f"median neighbors {np.median(candidate['neighbor_count']):.0f} | "
        f"median planarity {np.nanmedian(candidate['planarity']):.3f}"
    )

### Working neighborhood

For this sample, a 1.5 m radius is the initial working choice: it gives broad point coverage while retaining substantially smaller neighborhoods than the 2.0â€“2.5 m alternatives. This is an experimental parameter, not yet a production default.

In [ ]:
WORKING_RADIUS_M = 1.5
local_features = radius_comparison[WORKING_RADIUS_M]
feature_valid = local_features["valid"]
roof_x = roof_points[:, 0]
roof_y = roof_points[:, 1]
normal_z = np.clip(local_features["normals"][:, 2], 0.0, 1.0)
slope_degrees = np.degrees(np.arccos(normal_z))

fig, axes = plt.subplots(2, 2, figsize=(13, 11), constrained_layout=True)
feature_views = (
    ("Planarity", local_features["planarity"], "viridis"),
    ("Surface variation", local_features["surface_variation"], "magma"),
    ("Estimated slope (degrees)", slope_degrees, "cividis"),
    ("Neighborhood size", local_features["neighbor_count"], "plasma"),
)

for ax, (title, values, colormap) in zip(axes.flat, feature_views, strict=True):
    plot = ax.scatter(
        roof_x[feature_valid],
        roof_y[feature_valid],
        c=values[feature_valid],
        s=12,
        cmap=colormap,
    )
    ax.set(title=title, xlabel="Easting (m)", ylabel="Northing (m)")
    ax.set_aspect("equal")
    fig.colorbar(plot, ax=ax)

plt.show()
print(f"Valid local features: {feature_valid.sum():,} / {len(feature_valid):,}")

## 2. Independent experimental RANSAC baseline

This deliberately separate implementation supports research comparison with the bounded core diagnostic. It uses locally stable points and builds candidates for the later surface experiment; it must not be used by the production router. Thresholds remain explicit and experimental.

In [ ]:
def plane_residuals(points, coefficients):
    """Calculate orthogonal distances to z = ax + by + c."""
    a, b, c = coefficients
    vertical_residual = points[:, 2] - (a * points[:, 0] + b * points[:, 1] + c)
    return np.abs(vertical_residual) / np.sqrt(a**2 + b**2 + 1.0)


def fit_plane(points):
    """Fit z = ax + by + c by least squares."""
    design = np.column_stack((points[:, 0], points[:, 1], np.ones(len(points))))
    coefficients, _, _, _ = np.linalg.lstsq(design, points[:, 2], rcond=None)
    return coefficients


def segment_roof_planes(
    points,
    eligible,
    *,
    distance_threshold=0.25,
    min_points=40,
    max_planes=8,
    iterations=750,
    seed=42,
):
    """Extract dominant non-vertical planes with sequential RANSAC."""
    rng = np.random.default_rng(seed)
    labels = np.full(len(points), -1, dtype=np.int16)
    remaining = np.flatnonzero(eligible)
    models = []

    for plane_id in range(max_planes):
        if len(remaining) < min_points:
            break
        best_inliers = np.array([], dtype=np.int64)

        for _ in range(iterations):
            sample_indices = rng.choice(remaining, size=3, replace=False)
            sample = points[sample_indices]
            design = np.column_stack((sample[:, 0], sample[:, 1], np.ones(3)))
            if np.linalg.matrix_rank(design) < 3:
                continue
            candidate = fit_plane(sample)
            residuals = plane_residuals(points[remaining], candidate)
            inliers = remaining[residuals <= distance_threshold]
            if len(inliers) > len(best_inliers):
                best_inliers = inliers

        if len(best_inliers) < min_points:
            break

        model = fit_plane(points[best_inliers])
        refined_residuals = plane_residuals(points[remaining], model)
        inliers = remaining[refined_residuals <= distance_threshold]
        if len(inliers) < min_points:
            break
        labels[inliers] = plane_id
        models.append(model)
        remaining = remaining[refined_residuals > distance_threshold]

    return labels, models


xy_origin = roof_points[:, :2].mean(axis=0)
roof_local = np.column_stack(
    (roof_points[:, 0] - xy_origin[0], roof_points[:, 1] - xy_origin[1], roof_height)
)
stable = (
    feature_valid
    & (local_features["planarity"] >= 0.30)
    & (local_features["surface_variation"] <= 0.10)
)
plane_labels, plane_models = segment_roof_planes(roof_local, stable)
assigned = plane_labels >= 0

print(f"Geometrically stable points: {stable.sum():,} / {len(stable):,}")
print(f"Points assigned to planes: {assigned.sum():,} / {len(stable):,}")
print(f"Candidate planes: {len(plane_models)}")
for plane_id, model in enumerate(plane_models):
    plane_mask = plane_labels == plane_id
    slope = np.degrees(np.arctan(np.hypot(model[0], model[1])))
    residual = np.median(plane_residuals(roof_local[plane_mask], model))
    print(
        f"Plane {plane_id + 1}: {plane_mask.sum():,} points | "
        f"slope {slope:.1f} degrees | median residual {residual:.3f} m"
    )

In [ ]:
plane_palette = plt.colormaps["tab10"]
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(roof_x[~assigned], roof_y[~assigned], s=8, c="#bdbdbd", label="Unassigned")
for plane_id in range(len(plane_models)):
    mask = plane_labels == plane_id
    ax.scatter(
        roof_x[mask],
        roof_y[mask],
        s=12,
        color=plane_palette(plane_id),
        label=f"Plane {plane_id + 1}",
    )
ax.set(title="RANSAC roof-plane candidates", xlabel="Easting (m)", ylabel="Northing (m)")
ax.set_aspect("equal")
ax.legend(markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

plane_figure = go.Figure()
plane_figure.add_trace(
    go.Scatter3d(
        x=roof_x[~assigned],
        y=roof_y[~assigned],
        z=roof_height[~assigned],
        mode="markers",
        name="Unassigned",
        marker={"size": 2, "color": "#bdbdbd", "opacity": 0.5},
    )
)
for plane_id in range(len(plane_models)):
    mask = plane_labels == plane_id
    color = to_hex(plane_palette(plane_id))
    plane_figure.add_trace(
        go.Scatter3d(
            x=roof_x[mask],
            y=roof_y[mask],
            z=roof_height[mask],
            mode="markers",
            name=f"Plane {plane_id + 1}",
            marker={"size": 2.5, "color": color, "opacity": 0.85},
        )
    )
plane_figure.update_layout(
    title="Interactive roof-plane candidates",
    scene={
        "xaxis_title": "Easting (m)",
        "yaxis_title": "Northing (m)",
        "zaxis_title": "Height above ground (m)",
        "aspectmode": "data",
    },
    height=750,
)
plane_figure.show()

## 3. Split planes into connected regions and build candidate surfaces

RANSAC can merge disconnected but coplanar surfaces. A radius graph now separates spatial components. Each supported component is converted to a concave hull, clipped to the cadastral footprint, and lifted with its fitted plane equation. These polygons are diagnostic candidates rather than final roof topology.

In [ ]:
def spatial_components(xy, connection_radius):
    """Label connected components in a radius-neighbor graph."""
    pairs = cKDTree(xy).query_pairs(connection_radius, output_type="ndarray")
    if len(pairs) == 0:
        return np.arange(len(xy)), len(xy)
    rows = np.concatenate((pairs[:, 0], pairs[:, 1]))
    columns = np.concatenate((pairs[:, 1], pairs[:, 0]))
    graph = coo_matrix((np.ones(len(rows)), (rows, columns)), shape=(len(xy), len(xy)))
    component_count, labels = connected_components(graph, directed=False)
    return labels, component_count


CONNECTION_RADIUS_M = 1.5
MIN_REGION_POINTS = 20
region_labels = np.full(len(roof_points), -1, dtype=np.int16)
roof_regions = []

for plane_id, model in enumerate(plane_models):
    plane_indices = np.flatnonzero(plane_labels == plane_id)
    components, component_count = spatial_components(
        roof_points[plane_indices, :2], CONNECTION_RADIUS_M
    )
    for component_id in range(component_count):
        region_indices = plane_indices[components == component_id]
        if len(region_indices) < MIN_REGION_POINTS:
            continue
        region_id = len(roof_regions)
        region_labels[region_indices] = region_id
        roof_regions.append(
            {
                "region_id": region_id,
                "plane_id": plane_id,
                "model": model,
                "point_indices": region_indices,
            }
        )

print(f"Connected roof regions: {len(roof_regions)}")
for region in roof_regions:
    print(
        f"Region {region['region_id'] + 1}: plane {region['plane_id'] + 1} | "
        f"{len(region['point_indices']):,} points"
    )

In [ ]:
footprint_polygon = unary_union(
    [Polygon(rings[0], holes=rings[1:]) for rings in footprint_polygons]
)


def polygon_components(geometry):
    """Return every polygon nested in a Polygon, MultiPolygon, or collection."""
    if geometry.geom_type == "Polygon":
        return [geometry]
    return [
        polygon
        for component in getattr(geometry, "geoms", ())
        for polygon in polygon_components(component)
    ]


roof_surfaces = []

for region in roof_regions:
    region_xy = roof_points[region["point_indices"], :2]
    candidate = concave_hull(MultiPoint(region_xy), ratio=0.35)
    clipped = candidate.intersection(footprint_polygon)
    polygons = polygon_components(clipped)
    if not polygons:
        continue
    polygon = max(polygons, key=lambda geometry: geometry.area)
    roof_surfaces.append({**region, "polygon": polygon})

covered_area = unary_union([surface["polygon"] for surface in roof_surfaces]).area
print(f"Footprint covered by candidate surfaces: {covered_area:.1f} mÂ²")
print(f"Candidate surface coverage: {covered_area / footprint_polygon.area:.1%}")

fig, ax = plt.subplots(figsize=(10, 8))
for component_index, component in enumerate(polygon_components(footprint_polygon)):
    exterior_x, exterior_y = component.exterior.xy
    ax.plot(
        exterior_x,
        exterior_y,
        color="#222222",
        linewidth=2,
        label="Cadastral footprint" if component_index == 0 else None,
    )
    for interior in component.interiors:
        interior_x, interior_y = interior.xy
        ax.plot(interior_x, interior_y, color="#222222", linewidth=1, linestyle="--")
for surface in roof_surfaces:
    region_id = surface["region_id"]
    polygon_x, polygon_y = surface["polygon"].exterior.xy
    ax.fill(
        polygon_x,
        polygon_y,
        color=plane_palette(region_id),
        alpha=0.55,
        label=f"Region {region_id + 1} ({surface['polygon'].area:.1f} mÂ²)",
    )
ax.set(title="Candidate roof-surface polygons", xlabel="Easting (m)", ylabel="Northing (m)")
ax.set_aspect("equal")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

In [ ]:
surface_figure = go.Figure()
for surface in roof_surfaces:
    triangles = [
        triangle
        for triangle in triangulate(surface["polygon"])
        if surface["polygon"].covers(triangle.representative_point())
    ]
    vertices_xy = np.asarray(
        [coordinate for triangle in triangles for coordinate in list(triangle.exterior.coords)[:3]]
    )
    model = surface["model"]
    vertices_z = (
        model[0] * (vertices_xy[:, 0] - xy_origin[0])
        + model[1] * (vertices_xy[:, 1] - xy_origin[1])
        + model[2]
    )
    triangle_starts = np.arange(0, len(vertices_xy), 3)
    region_id = surface["region_id"]
    surface_figure.add_trace(
        go.Mesh3d(
            x=vertices_xy[:, 0],
            y=vertices_xy[:, 1],
            z=vertices_z,
            i=triangle_starts,
            j=triangle_starts + 1,
            k=triangle_starts + 2,
            name=f"Region {region_id + 1}",
            color=to_hex(plane_palette(region_id)),
            opacity=0.75,
            flatshading=True,
            showlegend=True,
        )
    )
surface_figure.update_layout(
    title="First candidate 3D roof surfaces",
    scene={
        "xaxis_title": "Easting (m)",
        "yaxis_title": "Northing (m)",
        "zaxis_title": "Height above ground (m)",
        "aspectmode": "data",
    },
    height=750,
)
surface_figure.show()

for surface in roof_surfaces:
    print(
        f"Region {surface['region_id'] + 1}: {surface['polygon'].area:.1f} mÂ² | "
        f"plane {surface['plane_id'] + 1}"
    )

## 4. Evaluate Roofer LoD2.2 against the experimental baseline

Roofer reconstructs a topologically complete solid and reports its own point-cloud and model-quality indicators. This section reads the generated CityJSONSeq directly, compares its coverage diagnostics with our partial surfaces, and visualizes the resulting semantic geometry. A successful process flag does not by itself imply acceptable geometric accuracy.

In [ ]:
roofer_root = PROJECT_ROOT / "outputs" / BUILDING_ID / "roofer"
native_output_directory = roofer_root / "native_cityjson"
roofer_output_directory = (
    native_output_directory if native_output_directory.exists() else roofer_root / "cityjson"
)
roofer_files = sorted(roofer_output_directory.glob("*.city.jsonl"))
assert roofer_files, f"No Roofer CityJSONSeq found in {roofer_output_directory}"

roofer_records = [
    json.loads(line)
    for line in roofer_files[0].read_text(encoding="utf-8").splitlines()
    if line.strip()
]
roofer_metadata = next(record for record in roofer_records if record["type"] == "CityJSON")
roofer_features = [record for record in roofer_records if record["type"] == "CityJSONFeature"]
roofer_reconstructions = []
for feature in roofer_features:
    building_object = feature["CityObjects"].get(BUILDING_ID)
    if building_object is None:
        continue
    for city_object in feature["CityObjects"].values():
        if city_object["type"] != "BuildingPart":
            continue
        for geometry in city_object.get("geometry", []):
            if geometry.get("lod") == "2.2":
                roofer_reconstructions.append(
                    {"feature": feature, "building": building_object, "lod22": geometry}
                )

assert roofer_reconstructions, "Roofer produced no BuildingPart with LoD 2.2"
representative_reconstruction = min(
    roofer_reconstructions,
    key=lambda item: item["building"]["attributes"]["rf_nodata_frac"],
)
roofer_building = representative_reconstruction["building"]
roofer_attributes = roofer_building["attributes"]
print(f"LoD2.2 reconstructed components: {len(roofer_reconstructions)} / {len(roofer_features)}")

comparison = {
    "process_succeeded": roofer_attributes["rf_success"],
    "extrusion_mode": roofer_attributes["rf_extrusion_mode"],
    "roof_type": roofer_attributes["rf_roof_type"],
    "detected_planes": roofer_attributes["rf_roof_planes"],
    "ridge_lines": roofer_attributes["rf_ridgelines"],
    "point_density_m2": roofer_attributes["rf_pt_density"],
    "nodata_fraction": roofer_attributes["rf_nodata_frac"],
    "lod22_rmse_m": roofer_attributes["rf_rmse_lod22"],
    "experimental_surface_coverage": covered_area / footprint_polygon.area,
}
for key, value in comparison.items():
    print(f"{key}: {value}")

quality_acceptable = (
    roofer_attributes["rf_success"]
    and roofer_attributes["rf_nodata_frac"] <= 0.20
    and roofer_attributes["rf_rmse_lod22"] <= 1.00
)
print(f"Accept as final LoD2.2 candidate: {quality_acceptable}")

### Common structural output quality

This is the first backend-independent benchmark gate: valid semantic solid, watertight boundary, plausible height, and coarse footprint agreement. Passing it does not yet mean the roof fits the LiDAR; point-to-surface accuracy is the next validation block.


In [ ]:
structural_quality = validate_cityjsonseq(roofer_files[0], footprint_polygons)
lidar_fit = assess_cityjson_lidar_fit(roofer_files[0], CROP_PATH, footprint_polygons)
final_quality = evaluate_reconstruction_quality(structural_quality, lidar_fit)
print(json.dumps(structural_quality.to_dict(), indent=2))
print(json.dumps(lidar_fit.to_dict(), indent=2))
print(json.dumps(final_quality.to_dict(), indent=2))
quality_labels = ["Geometry", "Watertight", "Plausibility", "Footprint bbox"]
quality_values = [
    float(structural_quality.geometry_valid),
    float(structural_quality.watertight),
    float(structural_quality.plausibility_pass),
    structural_quality.footprint_bbox_iou,
]
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(quality_labels, quality_values, color=["#2ca02c" if value >= 0.8 else "#d62728" for value in quality_values])
ax.bar_label(bars, fmt="%.3f")
ax.set(ylim=(0, 1.12), ylabel="Check value", title="Backend-independent structural QC")
plt.show()
print(f"Structural gate accepted: {structural_quality.accepted}")
print(f"Current LiDAR/Roofer heuristic accepted: {quality_acceptable}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
distance_names = ["Median", "RMSE", "P95"]
distance_values = [lidar_fit.distance_median_m, lidar_fit.distance_rmse_m, lidar_fit.distance_p95_m]
axes[0].bar(distance_names, distance_values, color="#d62728")
axes[0].set(ylabel="3D distance (m)", title="Observed roof to reconstructed surface")
support_names = ["<=0.2 m", "<=0.5 m", "<=1.0 m"]
support_values = [lidar_fit.within_020m_ratio, lidar_fit.within_050m_ratio, lidar_fit.within_100m_ratio]
axes[1].bar(support_names, support_values, color="#1f77b4")
axes[1].set(ylim=(0, 1), ylabel="Roof-point fraction", title="Metric support by tolerance")
plt.show()
print(f"FINAL DECISION: {'ACCEPTED' if final_quality.accepted else 'REJECTED'} ({final_quality.confidence_class.value})")
for failure in final_quality.failures:
    print(f"- {failure}")
roofer_benchmark_entry = build_benchmark_entry(
    BackendName.ROOFER, structural_quality, lidar_fit, final_quality
)
benchmark_report = compare_reconstructions((roofer_benchmark_entry,))
benchmark_columns = ["Backend", "LoD", "Accepted", "RMSE m", "P95 m", "<=0.5 m", "Faces"]
benchmark_rows = [[
    entry.backend.value, entry.lod, str(entry.accepted),
    f"{entry.point_surface_rmse_m:.2f}", f"{entry.point_surface_p95_m:.2f}",
    f"{entry.within_050m_ratio:.1%}", str(entry.face_count),
] for entry in benchmark_report.entries]
fig, ax = plt.subplots(figsize=(12, 2.2))
ax.axis("off")
table = ax.table(cellText=benchmark_rows, colLabels=benchmark_columns, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)
ax.set_title("Common reconstruction benchmark - implemented backends only")
plt.show()
print(f"Accepted ranking: {[backend.value for backend in benchmark_report.accepted_ranking] or 'none'}")
planned_fallback = reconstruction_plan.fallback_lod if not final_quality.accepted else None
print(f"Planned quality fallback: {planned_fallback or 'none'}")
if RUN_QUALITY_FALLBACK and not final_quality.accepted:
    primary_candidate = ReconstructionResult(
        status=ReconstructionStatus.SUCCESS,
        requested_lod=reconstruction_plan.requested_lod,
        targeted_lod=reconstruction_plan.target_lod,
        delivered_lod=reconstruction_plan.target_lod,
        backend=reconstruction_plan.selected_backend,
        model_path=roofer_files[0],
        provenance=GeometryProvenance(lidar_observed=True),
    )
    fallback_evidence = ReconstructionEvidence(
        BUILDING_ID, ROOFER_ROOT / "footprint_25830.geojson", lidar_points=CROP_PATH
    )
    fallback_result = finalize_or_execute_fallback(
        reconstruction_plan, fallback_evidence, backend_registry, primary_candidate, final_quality
    )
    print(f"Fallback status: {fallback_result.status.value}; target: {fallback_result.targeted_lod}")


In [ ]:
scale = np.asarray(roofer_metadata["transform"]["scale"])
translation = np.asarray(roofer_metadata["transform"]["translate"])
surface_colors = {"GroundSurface": "#8c564b", "WallSurface": "#7f7f7f", "RoofSurface": "#d62728"}
roofer_figure = go.Figure()
legend_types = set()

for reconstruction in roofer_reconstructions:
    feature = reconstruction["feature"]
    lod22 = reconstruction["lod22"]
    roofer_vertices = np.asarray(feature["vertices"]) * scale + translation
    semantic_surfaces = lod22["semantics"]["surfaces"]
    semantic_values = lod22["semantics"]["values"][0]
    for boundary, semantic_index in zip(lod22["boundaries"][0], semantic_values, strict=True):
        surface_type = semantic_surfaces[semantic_index]["type"]
        color = surface_colors[surface_type]
        for ring in boundary:
            closed_ring = [*ring, ring[0]]
            coordinates = roofer_vertices[closed_ring]
            roofer_figure.add_trace(
                go.Scatter3d(
                    x=coordinates[:, 0],
                    y=coordinates[:, 1],
                    z=coordinates[:, 2] - ground_z,
                    mode="lines",
                    name=surface_type,
                    legendgroup=surface_type,
                    showlegend=surface_type not in legend_types,
                    line={"color": color, "width": 5},
                )
            )
            legend_types.add(surface_type)

roofer_figure.update_layout(
    title="Roofer LoD2.2 semantic geometry",
    scene={
        "xaxis_title": "Easting (m)",
        "yaxis_title": "Northing (m)",
        "zaxis_title": "Height above local ground (m)",
        "aspectmode": "data",
    },
    height=750,
)
roofer_figure.show()

## 5. Evaluate the City3D OBJ candidate

City3D is evaluated with the same structural and LiDAR-fit thresholds as Roofer. The wrapper uses the median of classified ground returns instead of the raw point-cloud minimum. Because OBJ has no semantic labels, roof faces are inferred from orientation and elevation and this limitation is retained in the report.

In [ ]:
city3d_model = PROJECT_ROOT / "outputs" / BUILDING_ID / "city3d" / "building.obj"
if not city3d_model.exists():
    city3d_model = PROJECT_ROOT / "outputs" / BUILDING_ID / "city3d" / "building_python.obj"

if city3d_model.exists():
    city3d_geometry = validate_obj(city3d_model, footprint_polygons)
    city3d_lidar_fit = assess_obj_lidar_fit(city3d_model, CROP_PATH, footprint_polygons)
    city3d_quality = evaluate_reconstruction_quality(city3d_geometry, city3d_lidar_fit)
    print(json.dumps(city3d_geometry.to_dict(), indent=2))
    roofer_entry = build_benchmark_entry(BackendName.ROOFER, structural_quality, lidar_fit, final_quality)
    city3d_entry = build_benchmark_entry(BackendName.CITY3D, city3d_geometry, city3d_lidar_fit, city3d_quality)
    comparison = compare_reconstructions((roofer_entry, city3d_entry))
    print(json.dumps(comparison.to_dict(), indent=2))

    city3d_vertices, city3d_faces = read_obj(city3d_model)
    city3d_triangles = [
        (face[0], face[index], face[index + 1])
        for face in city3d_faces
        for index in range(1, len(face) - 1)
    ]
    city3d_figure = go.Figure(
        go.Mesh3d(
            x=city3d_vertices[:, 0], y=city3d_vertices[:, 1],
            z=city3d_vertices[:, 2] - city3d_vertices[:, 2].min(),
            i=[triangle[0] for triangle in city3d_triangles],
            j=[triangle[1] for triangle in city3d_triangles],
            k=[triangle[2] for triangle in city3d_triangles],
            color="#d97706", opacity=0.82, flatshading=True, name="City3D",
        )
    )
    city3d_figure.update_layout(
        title="City3D reconstructed OBJ", height=750,
        scene={"aspectmode": "data", "xaxis_title": "Easting (m)",
               "yaxis_title": "Northing (m)", "zaxis_title": "Relative height (m)"},
    )
    city3d_figure.show()
else:
    print(f"City3D model not found: {city3d_model}. Run scripts/run_city3d.py first.")

## Benchmark observations

Record conclusions here after running the cells:

- Is class 12 spatially redundant with the usable coverage?
- Are class-6 points clean inside the cadastral footprint?
- Does the ground median represent the local terrain adequately?
- Are multiple roof-height modes visible?
- Which artifacts should be removed before estimating roof planes?
- Does the 1.5 m neighborhood preserve visible roof boundaries?
- Which planarity and surface-variation ranges separate stable roof surfaces from edges?
- Do coplanar but disconnected roof regions need a spatial connectivity step?
- Are the 0.25 m distance and 40-point support thresholds appropriate for this density?
- Does the 1.5 m connectivity radius separate genuinely distinct surfaces?
- Where do concave hulls still differ from the visible roof boundaries?
- Which gaps require topology reconstruction rather than direct point outlining?
- Does Roofer's complete topology reflect actual roof structure or mostly fallback regularization?
- Is the reported LoD2.2 RMSE acceptable for the intended use?
- Should this building fall back to LoD1.x while denser samples continue to LoD2.2?